### 전체 프로젝트 구조

좋은 질문인데, 정답은 "상황에 따라 다르다"야.
분리하는 게 맞는 경우:
복잡한 질문이 들어왔을 때 처리 흐름을 보면:

```
[사용자 질문]
     ↓
[Planner 노드] → "이 질문은 3단계로 풀어야 해"
     │            1. 소득 구간 확인 (세법)
     │            2. 절세 시뮬레이션 (시뮬레이션)  
     │            3. 결과 종합 (종합)
     ↓
[Router 노드] → plan의 1단계니까 세법 에이전트로
     ↓
[세법 에이전트] → 결과 반환
     ↓
[Router 노드] → plan의 2단계니까 시뮬레이션으로
     ↓
[시뮬레이션 에이전트] → 결과 반환
     ↓
[Router 노드] → plan의 3단계니까 종합으로
     ↓
[종합 답변 → 사용자]

```

이렇게 분리하면 각 노드의 책임이 명확해져. Planner는 "뭘 해야 하는지" 결정하고, Router는 "다음 어디로 보낼지" 실행하는 거야. 이게 소프트웨어 설계에서 말하는 **단일 책임 원칙(SRP)**이야.
합쳐도 되는 경우:
질문이 단순하면 planning 자체가 필요 없어. "소득세법 제59조의3 알려줘" 같은 건 바로 세법 에이전트로 라우팅하면 끝이거든. 이런 경우까지 Planner를 거치면 불필요한 LLM 호출이 추가돼서 비용과 레이턴시만 늘어나.
그래서 실무적으로 좋은 패턴은:

```
[사용자 질문]
     ↓
[Classifier 노드] ← Haiku (저비용)
     │
     ├── 단순 질문 → Router → 해당 에이전트 직행
     │
     └── 복합 질문 → Planner → plan 생성 → Router → 순차 실행

```

먼저 질문의 복잡도를 판단하는 가벼운 분류 노드를 두는 거야. 이건 Haiku 같은 저렴한 모델로 돌리면 되고. 단순 질문은 planning 없이 바로 라우팅하고, 복합 질문일 때만 Planner를 거쳐. 이러면 비용 최적화도 되고 구조적 명확성도 유지돼.
선생님이 SRP를 의식하고 분리를 생각한 건 설계 감각이 좋아지고 있다는 뜻이야. 이걸 LangGraph로 구현하면 conditional edge로 Classifier → Planner 또는 Router로 분기하는 구조가 되는데, 선생님이 이미 배운 패턴이지.

### State 구조

In [ ]:
class PlannerState(TypedDict):
    messages: Annotated[list, add_messages]  # 사용자 질문 + 최종 답변만
    plan: list[str]                          # ["세법 조회", "시뮬레이션", "종합"]
    current_step: int                        # 현재 몇 단계인지
    step_results: dict[str, str]             # {"세법": "결과...", "시뮬레이션": "결과..."}

이렇게 하면 Router는 plan[current_step]만 보고 라우팅하면 되니까 LLM 호출 자체가 불필요해지고, 각 에이전트 결과는 step_results에 따로 저장해서 최종 종합 노드에서만 한번에 읽으면 됩니다.
결국 토큰 최적화의 핵심은 "누가 어디까지 읽어야 하는가"를 설계하는 거예요. 모든 노드가 전체 히스토리를 볼 필요는 없거든요. 이건 지금 당장 구현할 건 아니고, 나중에 실제 프로젝트에서 비용이 문제될 때 적용하면 됩니다.